# 12.3 A small molecular GNN experiment: measured aqueous solubility

Message passing becomes useful when it is connected to a well-defined prediction question. Here we build a complete, deliberately small experiment: **Can a two-layer molecular GNN predict measured log solubility for held-out scaffold groups?** We compare it with a training mean, a descriptor model, and a Morgan-fingerprint model on exactly the same rows.

## Learning objectives

1. Connect atom/bond tensors to a measured graph-level target.
2. Keep chemical groups and fitted preprocessing out of the wrong partitions.
3. Train a compact GNN, select a validation checkpoint, and evaluate a frozen experiment.
4. Compare appropriate baselines and save enough information to reproduce inference.
5. Recognize the difference between a short teaching experiment and a benchmark study.

**Prerequisites:** Chapters 10–11 and [12.2: message passing](Chapter12_Part2.ipynb). **Runtime:** CPU, one thread, at most 400 small molecules and 80 full-batch training steps; no download or prior notebook output. Use the [course environment](Readme.md#set-up-python), then restart and run all cells in order. All graph code is visible below; no graph framework is required.

[Previous: message passing](Chapter12_Part2.ipynb) · [Chapter contents](Readme.md) · [Next: diagnostics and explanations](Chapter12_Part4.ipynb)

### Start here: one experimental question, three separate jobs

A **feature** is information supplied to the model; a **target** is the measured quantity it should predict. Training rows adjust model parameters. Validation rows choose a checkpoint. Test rows assess the frozen procedure. A mini-batch is only a group processed together during computation; it is different from these experimental partitions.

Our target is logarithmic: $y=-3$ means $S=10^{-3}$ mol/L, or 1 mmol/L. If prediction minus measurement is $+1$, predicted solubility is ten times the measured value; $-1$ means one tenth. The numerical loss works in log units because the measured concentrations span orders of magnitude.

**Research question:** can a model help prioritize compounds for a solubility assay? Before fitting, record the chemical domain, measurement definition, partition rule, baseline, and evaluation metric. This notebook supplies that workflow, then a short post-assessment exercise translates errors into a concentration decision. **First pass:** follow the audit, split, learning curves, and baseline comparison. **Deeper pass:** graph internals and checkpoint reconstruction. [Part 7](Chapter12_Part7.ipynb) repeats the same data split using PyG so implementation choices can be compared fairly.

In [ ]:
import os
os.environ["MKL_THREADING_LAYER"] = "SEQUENTIAL"
for key in ("OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS"):
    os.environ[key] = "1"

from pathlib import Path
import hashlib
import json
import platform
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
import sklearn
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from rdkit import Chem, rdBase, DataStructs
from rdkit.Chem import Descriptors, rdFingerprintGenerator
from rdkit.Chem.Scaffolds import MurckoScaffold
from IPython.display import display

SEED = 2026
torch.set_num_threads(1)
torch.use_deterministic_algorithms(True)
torch.manual_seed(SEED)
OUT = Path("outputs/chapter12_part3")
OUT.mkdir(parents=True, exist_ok=True)
VERSIONS = {"python": platform.python_version(), "torch": str(torch.__version__),
            "rdkit": rdBase.rdkitVersion, "numpy": np.__version__,
            "sklearn": sklearn.__version__}
print(VERSIONS)

## 12.3.1 Specify the target and audit the source

The bundled table contains 1,121 observations corresponding to the **measured**, not ESOL-predicted, column of the Delaney snapshot distributed by DeepChem. The target is

$$
y=\log_{10}\!\left(\frac{S}{1\ \mathrm{mol\,L^{-1}}}\right).
$$

A one-unit difference means a tenfold concentration ratio for an individual pair of values. Temperature, pH, and measurement details are missing from this export; a graph cannot supply those conditions. See the [dataset provenance and comparison](datasets/README.md) and [original ESOL paper](https://doi.org/10.1021/ci034243x).

We audit every row, then select at most 400 connected molecules with 1–60 heavy atoms using a **label-independent SMILES hash**. This bound controls runtime; it changes the population being studied. Repeated measurements are retained and will stay in the same partition. We do not average distinct measured values or silently neutralize molecules.

In [ ]:
data_path = Path("datasets/Solubility.csv")
source_sha256 = hashlib.sha256(data_path.read_bytes()).hexdigest()
assert source_sha256 == "3fedd5ad80f9231bd331929ba0943a117d0d6ee3f75eda9a27fab9b4ab974ad5", "Review changed data first."
raw = pd.read_csv(data_path)
assert list(raw.columns) == ["smiles", "solubility"]
assert raw.notna().all().all() and np.isfinite(raw["solubility"]).all()
raw["source_row"] = np.arange(len(raw))  # zero-based row after the header
raw["smiles"] = raw["smiles"].str.strip()
all_mols = [Chem.MolFromSmiles(s) for s in raw["smiles"]]
invalid = [i for i, m in enumerate(all_mols) if m is None or m.GetNumAtoms() == 0]
assert not invalid, f"Unusable source rows: {invalid}"
raw["canonical_smiles"] = [Chem.MolToSmiles(m, isomericSmiles=True) for m in all_mols]
raw["heavy_atoms"] = [m.GetNumHeavyAtoms() for m in all_mols]
raw["fragments"] = [len(Chem.GetMolFrags(m)) for m in all_mols]
eligible = raw["fragments"].eq(1) & raw["heavy_atoms"].between(1, 60)
raw["selection_key"] = raw["smiles"].map(lambda s: hashlib.sha256(s.encode()).hexdigest())
sample = raw.loc[eligible].sort_values(["selection_key", "source_row"]).head(400).copy().reset_index(drop=True)
molecules = [all_mols[i] for i in sample["source_row"]]
identity = raw.groupby("canonical_smiles")["solubility"].agg(["size", "nunique"])
print({"source_rows": len(raw), "invalid": len(invalid), "eligible": int(eligible.sum()),
       "selected": len(sample), "duplicate_extra_rows_in_source": int((identity["size"] - 1).sum()),
       "structure_groups_with_distinct_targets": int(identity["nunique"].gt(1).sum())})

## 12.3.2 Split chemical groups before fitting anything

We use exact, non-stereochemical Bemis–Murcko scaffolds as groups and hash each group into train/validation/test with nominal probabilities 70/15/15. This is a **fixed teaching split**, not an official MoleculeNet split. Hash thresholds do not guarantee those row fractions: large groups can dominate. All acyclic structures have an empty scaffold, so we deliberately place them in one `ACYCLIC` group. Report that count rather than interpreting this split as a representative random sample.

No exact group or canonical structure may span partitions. Different Murcko groups can still be very similar; a scaffold split does not guarantee that every test compound is far from training chemistry. In a deployment study, choose temporal, project, series, or other grouping that matches the intended use. Repeats within a partition receive additional weight and are not independent measurements of generalization.

In [ ]:
sample["group"] = [MurckoScaffold.MurckoScaffoldSmiles(mol=m, includeChirality=False)
                   or "ACYCLIC" for m in molecules]
def partition_for(group):
    fraction = int(hashlib.sha256(f"{SEED}|{group}".encode()).hexdigest()[:8], 16) / 2**32
    return "train" if fraction < 0.70 else "validation" if fraction < 0.85 else "test"

sample["partition"] = sample["group"].map(partition_for)
partitions = ["train", "validation", "test"]
indices = {p: np.flatnonzero(sample["partition"].eq(p)) for p in partitions}
train_id, val_id, test_id = [indices[p] for p in partitions]
assert all(len(indices[p]) >= 15 for p in partitions)
assert sample.groupby("group")["partition"].nunique().max() == 1
assert sample.groupby("canonical_smiles")["partition"].nunique().max() == 1
display(sample.groupby("partition").agg(rows=("source_row", "size"),
    groups=("group", "nunique"), acyclic_rows=("group", lambda s: s.eq("ACYCLIC").sum())))
sample.drop(columns="selection_key").to_csv(OUT / "split.csv", index=False)

## 12.3.3 Convert molecules to graph tensors

Element identity is categorical (one-hot with an `other` slot). Degree, attached H count, and formal charge are counts divided by fixed constants, followed by aromatic/ring flags. These constants are part of the encoding and are **not estimated from the dataset**. Every bond contributes two directed edges with identical bond attributes: a five-way bond-type encoding including `other`, conjugation, and ring membership.

This compact encoding preserves formal charges and recorded connectivity but omits stereochemistry, isotope masses, radical electrons, coordinates, and experimental conditions. Unusual chemistry can map to the same `other` slot. We audit omitted features and unknown elements below; a tensor of the right shape is not proof that a model is applicable. Hydrogen atoms are normally implicit under RDKit's default SMILES parser, with their attached counts included on heavy atoms. Our single-molecule inference rejects disconnected inputs consistently with the training selection.

In [ ]:
ELEMENTS = [5, 6, 7, 8, 9, 14, 15, 16, 17, 35, 53]
BOND_TYPES = [Chem.BondType.SINGLE, Chem.BondType.DOUBLE,
              Chem.BondType.TRIPLE, Chem.BondType.AROMATIC]
NODE_NAMES = [f"element_{z}" for z in ELEMENTS] + ["element_other", "degree/4",
    "attached_H/4", "formal_charge/2", "aromatic", "in_ring"]
EDGE_NAMES = ["single", "double", "triple", "aromatic", "other", "conjugated", "in_ring"]

def one_hot_other(value, choices):
    return [float(value == c) for c in choices] + [float(value not in choices)]

def molecular_graph(mol):
    if mol is None or mol.GetNumAtoms() == 0 or len(Chem.GetMolFrags(mol)) != 1:
        raise ValueError("Expected a parsed, nonempty, connected molecule.")
    nodes = [one_hot_other(a.GetAtomicNum(), ELEMENTS) +
             [a.GetDegree()/4, a.GetTotalNumHs()/4, a.GetFormalCharge()/2,
              float(a.GetIsAromatic()), float(a.IsInRing())] for a in mol.GetAtoms()]
    edges, features = [], []
    for bond in mol.GetBonds():
        i, j = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
        attr = one_hot_other(bond.GetBondType(), BOND_TYPES) + [float(bond.GetIsConjugated()), float(bond.IsInRing())]
        edges.extend([(i, j), (j, i)])
        features.extend([attr, attr])
    return {"x": torch.tensor(nodes, dtype=torch.float32),
            "edge_index": torch.tensor(edges, dtype=torch.long).reshape(-1, 2).T.contiguous(),
            "edge_attr": torch.tensor(features, dtype=torch.float32).reshape(-1, len(EDGE_NAMES))}

def batch_graphs(graph_list):
    if not graph_list:
        raise ValueError("A batch must contain at least one graph.")
    offsets = np.cumsum([0] + [len(g["x"]) for g in graph_list[:-1]])
    packed = {"x": torch.cat([g["x"] for g in graph_list]),
              "edge_index": torch.cat([g["edge_index"] + int(o) for g, o in zip(graph_list, offsets)], dim=1),
              "edge_attr": torch.cat([g["edge_attr"] for g in graph_list]),
              "batch": torch.cat([torch.full((len(g["x"]),), i, dtype=torch.long) for i, g in enumerate(graph_list)]),
              "n_graphs": len(graph_list)}
    src, dst = packed["edge_index"]
    assert torch.equal(packed["batch"][src], packed["batch"][dst]), "An edge crosses molecular graphs."
    return packed

graphs = [molecular_graph(m) for m in molecules]
batches = {p: batch_graphs([graphs[i] for i in indices[p]]) for p in partitions}
water = molecular_graph(Chem.MolFromSmiles("O"))
assert water["edge_index"].shape == (2, 0) and water["edge_attr"].shape == (0, len(EDGE_NAMES))
encoding_audit = {"unknown_element_atoms": sum(a.GetAtomicNum() not in ELEMENTS for m in molecules for a in m.GetAtoms()),
    "isotope_atoms": sum(a.GetIsotope() != 0 for m in molecules for a in m.GetAtoms()),
    "radical_atoms": sum(a.GetNumRadicalElectrons() != 0 for m in molecules for a in m.GetAtoms()),
    "specified_chiral_atoms": sum(a.GetChiralTag() != Chem.ChiralType.CHI_UNSPECIFIED for m in molecules for a in m.GetAtoms())}
print("Encoding audit:", encoding_audit)
print("Training nodes / directed edges:", tuple(batches["train"]["x"].shape), tuple(batches["train"]["edge_index"].shape))

## 12.3.4 A compact bond-aware MPNN

For directed edge $j\to i$, round $t$ uses

$$
m_{j\to i}^{(t)}=\mathrm{ReLU}(W_h^{(t)}h_j^{(t)}+W_e^{(t)}e_{ji}),\quad
a_i^{(t)}=\frac{1}{\max(1,|\mathcal N(i)|)}\sum_{j\in\mathcal N(i)}m_{j\to i}^{(t)},
$$
$$
h_i^{(t+1)}=h_i^{(t)}+\mathrm{ReLU}\!\left(W_u^{(t)}[h_i^{(t)}\Vert a_i^{(t)}]+b_u^{(t)}\right).
$$

Weights are shared across atoms/edges within a round, with distinct parameters in each round. Two rounds incorporate information through two bond hops. The graph readout concatenates **mean atom embeddings and $\log(1+N)$**, then uses a small regression head. The explicit atom count avoids losing size information in mean pooling. It does not enforce extensivity; log solubility is not an additive energy.

This is a teaching MPNN, not a reproduction of Chemprop or a named published architecture. [Gilmer et al.](https://proceedings.mlr.press/v70/gilmer17a.html) gives the general message-passing framework; [Yang et al.](https://arxiv.org/abs/1904.01561) evaluates molecular representations and directed-bond models.

In [ ]:
class SolubilityMPNN(nn.Module):
    def __init__(self, node_dim, edge_dim, hidden=24, rounds=2):
        super().__init__()
        self.encoder = nn.Linear(node_dim, hidden)
        self.node_messages = nn.ModuleList([nn.Linear(hidden, hidden, bias=False) for _ in range(rounds)])
        self.bond_messages = nn.ModuleList([nn.Linear(edge_dim, hidden, bias=False) for _ in range(rounds)])
        self.updates = nn.ModuleList([nn.Linear(2*hidden, hidden) for _ in range(rounds)])
        self.head = nn.Sequential(nn.Linear(hidden+1, hidden), nn.ReLU(), nn.Linear(hidden, 1))

    def forward(self, graph):
        h = torch.relu(self.encoder(graph["x"]))
        src, dst = graph["edge_index"]
        degree = torch.bincount(dst, minlength=len(h)).to(h.dtype).clamp_min(1).unsqueeze(1)
        for node_map, bond_map, update in zip(self.node_messages, self.bond_messages, self.updates):
            messages = torch.relu(node_map(h[src]) + bond_map(graph["edge_attr"]))
            aggregate = torch.zeros_like(h).index_add_(0, dst, messages) / degree
            h = h + torch.relu(update(torch.cat([h, aggregate], dim=1)))
        counts = torch.bincount(graph["batch"], minlength=graph["n_graphs"]).to(h.dtype).unsqueeze(1)
        pooled = h.new_zeros((graph["n_graphs"], h.shape[1])).index_add_(0, graph["batch"], h) / counts
        return self.head(torch.cat([pooled, torch.log1p(counts)], dim=1)).squeeze(1)

ARCHITECTURE = {"node_dim": len(NODE_NAMES), "edge_dim": len(EDGE_NAMES), "hidden": 24, "rounds": 2}
torch.manual_seed(SEED)
model = SolubilityMPNN(**ARCHITECTURE)
print("Trainable parameters:", sum(p.numel() for p in model.parameters()))
assert model(batches["train"]).shape == (len(train_id),)

## 12.3.5 Predeclare the training protocol

Only **training labels** determine the target mean and standard deviation. All models use the same training observations. The GNN has a fixed architecture, Adam optimizer (learning rate 0.01), at most 80 epochs, and patience 12. One epoch is one full-batch update: this fits in memory because the teaching subset is small. Larger datasets need shuffled mini-batches containing whole molecular graphs, never independently sampled atoms with a repeated molecular label.

Validation MSE selects the lowest-loss checkpoint; patience uses a separate improvement tolerance. The actual best weights are restored before inference. We do not search architectures or seeds after looking at test scores. Baseline settings below are fixed in advance, so this comparison measures these concrete recipes, not the best possible version of each model family.

In [ ]:
y = sample["solubility"].to_numpy(dtype=np.float64)
y_mean, y_std = float(y[train_id].mean()), float(y[train_id].std())
assert y_std > 0
y_train = torch.tensor((y[train_id]-y_mean)/y_std, dtype=torch.float32)
y_val = torch.tensor((y[val_id]-y_mean)/y_std, dtype=torch.float32)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
loss_function = nn.MSELoss()
MAX_EPOCHS, PATIENCE, MIN_DELTA = 80, 12, 1e-4
best_loss, patience_reference = float("inf"), float("inf")
best_state, best_epoch, stale_epochs = None, None, 0
history_rows = []

In [ ]:
training_start = time.perf_counter()
for epoch in range(1, MAX_EPOCHS+1):
    model.train()
    optimizer.zero_grad(set_to_none=True)
    loss = loss_function(model(batches["train"]), y_train)
    loss.backward()
    optimizer.step()
    model.eval()
    with torch.inference_mode():
        train_loss = loss_function(model(batches["train"]), y_train).item()
        val_loss = loss_function(model(batches["validation"]), y_val).item()
    if not np.isfinite([train_loss, val_loss]).all():
        raise RuntimeError("Non-finite loss; inspect inputs and optimizer before evaluating.")
    history_rows.append({"epoch": epoch, "train_rmse": np.sqrt(train_loss)*y_std,
                         "validation_rmse": np.sqrt(val_loss)*y_std})
    if val_loss < best_loss:
        best_loss, best_epoch = val_loss, epoch
        best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
    if val_loss < patience_reference - MIN_DELTA:
        patience_reference, stale_epochs = val_loss, 0
    else:
        stale_epochs += 1
    if stale_epochs >= PATIENCE:
        break
model.load_state_dict(best_state)
model.eval()
with torch.inference_mode():
    restored_validation = loss_function(model(batches["validation"]), y_val).item()
assert np.isclose(restored_validation, best_loss, rtol=1e-6)
training_seconds = time.perf_counter() - training_start
history = pd.DataFrame(history_rows)
history.to_csv(OUT / "learning_history.csv", index=False)
print(f"{epoch} epochs in {training_seconds:.2f} s; restored epoch {best_epoch}.")
print(f"Selected validation RMSE: {np.sqrt(best_loss)*y_std:.3f} log units")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.6), layout="constrained")
ax.plot(history["epoch"], history["train_rmse"], label="Training")
ax.plot(history["epoch"], history["validation_rmse"], label="Validation")
ax.axvline(best_epoch, color="black", linestyle=":", label=f"Restored epoch {best_epoch}")
ax.set(xlabel="Epoch", ylabel="RMSE (log solubility units)", title="Validation selects the checkpoint")
ax.legend()
fig.savefig(OUT / "learning_curves.png", dpi=150)
plt.show()

## 12.3.6 Test the implementation before judging its chemistry

Permutation invariance and graph isolation are architectural requirements. A model can satisfy them and still predict poorly. We verify them using validation molecules: reversing an atom order must preserve its prediction, and batching two molecules must agree with processing them separately. Floating-point reductions may change roundoff when their summation order changes, so comparisons use a small tolerance.

In [ ]:
probe_mols = [molecules[i] for i in val_id[:2]]
renumbered = Chem.RenumberAtoms(probe_mols[0], list(reversed(range(probe_mols[0].GetNumAtoms()))))
with torch.inference_mode():
    together = model(batch_graphs([molecular_graph(m) for m in probe_mols]))
    separate = torch.cat([model(batch_graphs([molecular_graph(m)])) for m in probe_mols])
    permuted = model(batch_graphs([molecular_graph(renumbered)]))[0]
torch.testing.assert_close(together, separate, atol=2e-6, rtol=2e-6)
torch.testing.assert_close(together[0], permuted, atol=2e-6, rtol=2e-6)
print("Permutation invariance and separate-versus-batched inference passed.")

## 12.3.7 Fit useful baselines on the same training rows

* **Training mean:** no structural information; a basic check on learnability.
* **Descriptor ridge:** nine named RDKit descriptors with a training-fitted `StandardScaler`, then `Ridge(alpha=1)`.
* **Morgan random forest:** radius-2, 1,024-bit fingerprints and 64 trees, maximum depth 12, one CPU thread. The fingerprint includes no chirality, matching this GNN's omission.

Descriptors include calculated log P and polar surface area; they are not measured solubility labels. They supply useful chemical prior information. Fingerprints also summarize neighborhoods, using fixed algorithms rather than learned message functions. A competitive baseline is part of understanding what the GNN learns, not an obstacle to teaching it.

In [ ]:
DESCRIPTOR_NAMES = ["MolWt", "NumHeteroatoms", "RingCount", "NumHAcceptors", "NumHDonors",
                    "FractionCSP3", "TPSA", "MolLogP", "MolMR"]
descriptor_functions = dict(Descriptors.descList)
X_desc = np.array([[descriptor_functions[name](m) for name in DESCRIPTOR_NAMES] for m in molecules])
assert np.isfinite(X_desc).all()
fp_generator = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=1024, includeChirality=False)
fingerprints = [fp_generator.GetFingerprint(m) for m in molecules]
X_fp = np.asarray([fp_generator.GetFingerprintAsNumPy(m) for m in molecules], dtype=np.float32)
ridge = make_pipeline(StandardScaler(), Ridge(alpha=1.0))
forest = RandomForestRegressor(n_estimators=64, max_depth=12, random_state=SEED, n_jobs=1)
ridge.fit(X_desc[train_id], y[train_id])
forest.fit(X_fp[train_id], y[train_id])
print("Fitted descriptor ridge and Morgan random forest using training rows only.")

## 12.3.8 Evaluate the frozen recipes once

Now access test labels for final assessment. MAE and RMSE are in log solubility units; RMSE penalizes large errors more strongly. $R^2$ can be negative and does not establish calibration. The scatter plot is descriptive, not a confidence interval. We also report a predeclared acyclic-versus-cyclic breakdown to expose the unusual held-out group composition.

These are small, single-seed, non-official subsets with uneven scaffold groups. A score difference does not establish a general ranking of GNNs, fingerprints, and descriptors. The 400-row selection differs from Chapter 11.3's 600-row example, so compare the models **inside this notebook**, not their headline scores across chapters. A larger study should predefine repeated group splits, tune each model comparably, and quantify uncertainty using chemically meaningful groups.

In [ ]:
with torch.inference_mode():
    gnn_test = model(batches["test"]).numpy()*y_std + y_mean
predictions = {"Training mean": np.full(len(test_id), y_mean),
               "Descriptor ridge": ridge.predict(X_desc[test_id]),
               "Morgan forest": forest.predict(X_fp[test_id]), "GNN": gnn_test}
def regression_metrics(observed, predicted):
    return {"MAE": mean_absolute_error(observed, predicted),
            "RMSE": np.sqrt(mean_squared_error(observed, predicted)),
            "R2": r2_score(observed, predicted)}

metrics = pd.DataFrame({name: regression_metrics(y[test_id], pred) for name, pred in predictions.items()}).T
display(metrics.round(3))
test_table = sample.iloc[test_id][["source_row", "canonical_smiles", "group", "solubility"]].copy()
for name, pred in predictions.items():
    test_table[name] = pred
test_table["GNN_absolute_error"] = abs(test_table["GNN"]-test_table["solubility"])
test_table["acyclic"] = test_table["group"].eq("ACYCLIC")
display(test_table.groupby("acyclic").agg(rows=("source_row", "size"),
    scaffold_groups=("group", "nunique"), GNN_MAE=("GNN_absolute_error", "mean")))
metrics.to_csv(OUT / "test_metrics.csv")
test_table.to_csv(OUT / "test_predictions.csv", index=False)

### Read the result in its data context

With the bundled snapshot, there are 245 training rows, only 19 validation rows, and 136 test rows. Of those test rows, **111 belong to the single acyclic group**. The small validation sample makes checkpoint selection noisy, and the row-level test metrics give that one group considerable weight. These limitations are consequences of the stated split and teaching budget; they are not hidden behind the aggregate score.

In the tested environment, descriptor ridge has lower test RMSE than this small GNN. That is a valid outcome of the experiment. Do not respond by repeatedly changing the GNN until it wins on the same test labels. A prospective follow-up would specify its changes and evaluation design before inspecting a new held-out assessment.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4), layout="constrained")
limits = [min(y[test_id].min(), min(p.min() for p in predictions.values()))-0.3,
          max(y[test_id].max(), max(p.max() for p in predictions.values()))+0.3]
for ax, name in zip(axes, ["GNN", "Descriptor ridge", "Morgan forest"]):
    ax.scatter(y[test_id], predictions[name], s=22, alpha=0.65, edgecolors="none")
    ax.plot(limits, limits, "k--", linewidth=1)
    ax.set(xlabel="Measured log solubility", ylabel="Predicted log solubility",
           title=f"{name}: held-out scaffold groups", xlim=limits, ylim=limits)
fig.savefig(OUT / "test_parity.png", dpi=150)
plt.show()

### Similarity is context, not a calibrated uncertainty estimate

For each test molecule, find the maximum binary Morgan Tanimoto similarity to any training molecule. This calculation uses structures only. Similarity depends on the encoding and is not a probability of being correct; low similarity can flag an unsupported query, while high similarity does not protect against an activity cliff or missing experimental conditions. We save the values for inspection without selecting a threshold or changing the model after viewing the test errors.

In [ ]:
train_fps = [fingerprints[i] for i in train_id]
nearest_similarity = np.array([max(DataStructs.BulkTanimotoSimilarity(fingerprints[i], train_fps)) for i in test_id])
test_table["max_training_Tanimoto"] = nearest_similarity
test_table.to_csv(OUT / "test_predictions.csv", index=False)
fig, ax = plt.subplots(figsize=(6.5, 3.6), layout="constrained")
ax.scatter(nearest_similarity, abs(gnn_test-y[test_id]), s=22, alpha=0.65)
ax.set(xlabel="Maximum training Morgan Tanimoto", ylabel="Absolute GNN error (log units)",
       title="Structural similarity supplies context, not error bars", xlim=(0, 1.03))
fig.savefig(OUT / "similarity_error.png", dpi=150)
plt.show()

### Worked research interpretation: would a 1 mmol/L solubility screen be reliable?

Suppose an assay needs a dissolved concentration of 1 mmol/L. That corresponds to a log-solubility threshold of $-3$. For this **post-assessment teaching scenario**, apply the same stated threshold to measured and predicted solubility. We do not optimize the threshold or refit any model using these test labels.

A false positive means predicted solubility clears the chosen concentration while the recorded measurement does not; a false negative means a potentially suitable compound is deprioritized. Their practical costs depend on the experiment. The cumulative error curve translates $|\hat y-y|$ into the multiplicative discrepancy $10^{|\hat y-y|}$: a one-log-unit error is a factor of ten.

These historical solubility measurements do not specify every intended assay condition. This is an illustration of a decision rule, not proof of solubility at a new pH, temperature, or formulation. Discuss what measurements would be needed before running that assay. This additional reading of the held-out results creates no new independent validation set.

In [ ]:
from sklearn.metrics import confusion_matrix
concentration_threshold = -3.0
measured_clears = y[test_id] >= concentration_threshold
fig, axes = plt.subplots(1,3,figsize=(12,3.7),layout='constrained')
decision_rows = []
for ax,name in zip(axes[:2], ['GNN','Descriptor ridge']):
    predicted_clears = predictions[name] >= concentration_threshold
    counts = confusion_matrix(measured_clears,predicted_clears,labels=[False,True])
    ax.imshow(counts,cmap='Blues',vmin=0)
    for (row,col),value in np.ndenumerate(counts):
        ax.text(col,row,str(value),ha='center',va='center',
                color='white' if value>counts.max()/2 else 'black')
    ax.set(xticks=[0,1],yticks=[0,1],xticklabels=['Below','Clears'],yticklabels=['Below','Clears'],
           xlabel='Predicted versus 1 mmol/L',ylabel='Measured versus 1 mmol/L',title=name)
    decision_rows.append({'model':name,'threshold_logS':concentration_threshold,
                          'TN':counts[0,0],'FP':counts[0,1],'FN':counts[1,0],'TP':counts[1,1]})
    error = np.sort(np.abs(predictions[name]-y[test_id]))
    axes[2].step(error,np.arange(1,len(error)+1)/len(error),where='post',label=name)
for value in [.30103,1.,2.]:
    axes[2].axvline(value,color='.8',lw=.7)
axes[2].set(xlabel='Absolute log error\n0.301 ≈ 2×; 1 = 10×; 2 = 100×',
            ylabel='Fraction of test rows within error',ylim=(0,1.03),title='Reading error in concentration units')
axes[2].legend(fontsize=8)
pd.DataFrame(decision_rows).to_csv(OUT / 'concentration_decision_example.csv',index=False)
fig.savefig(OUT / 'solubility_research_decision.png',dpi=150)
plt.show()

## 12.3.9 Save the representation with the weights

Weights alone do not identify a molecular model. Inference also needs the feature order and categories, hydrogen/fragment/stereo policy, architecture, target scale, software versions, and training data/split identifiers. This checkpoint is **for inference**, not exact optimizer-state resumption. The notebook contains the implementation; the accompanying split table and JSON record preserve the experiment.

We load the checkpoint with `weights_only=True`, reconstruct the model, and require its predictions on two validation molecules to match. This validates serialization, not external chemical accuracy. Only load model files whose origin you trust.

In [ ]:
FEATURE_SCHEMA = {"node_names": NODE_NAMES, "edge_names": EDGE_NAMES, "elements": ELEMENTS,
    "bond_types": [str(b) for b in BOND_TYPES], "hydrogens": "RDKit default implicit H with attached counts",
    "fragments": "one connected component", "stereo": "omitted", "coordinates": "omitted"}
checkpoint = {"state_dict": model.state_dict(), "architecture": ARCHITECTURE,
    "feature_schema": FEATURE_SCHEMA, "target_mean": y_mean, "target_std": y_std,
    "target": "log10(S / (1 mol/L)); measured aqueous solubility",
    "source_sha256": source_sha256, "train_source_rows": sample.iloc[train_id]["source_row"].tolist(),
    "validation_source_rows": sample.iloc[val_id]["source_row"].tolist(), "seed": SEED,
    "best_epoch": best_epoch, "versions": VERSIONS}
checkpoint_path = OUT / "solubility_mpnn.pt"
torch.save(checkpoint, checkpoint_path)
loaded = torch.load(checkpoint_path, map_location="cpu", weights_only=True)
assert loaded["feature_schema"] == FEATURE_SCHEMA
restored = SolubilityMPNN(**loaded["architecture"])
restored.load_state_dict(loaded["state_dict"])
restored.eval()

def predict_smiles(smiles_list):
    mols = [Chem.MolFromSmiles(s) for s in smiles_list]
    packed = batch_graphs([molecular_graph(m) for m in mols])
    with torch.inference_mode():
        values = restored(packed).numpy()*loaded["target_std"] + loaded["target_mean"]
    return values  # no calibrated intervals; valid parsing does not establish applicability

query_smiles = sample.iloc[val_id[:2]]["smiles"].tolist()
roundtrip = predict_smiles(query_smiles)
np.testing.assert_allclose(roundtrip, together.numpy()*y_std+y_mean, atol=1e-5, rtol=1e-5)
record = {"scope": "400-row bounded teaching experiment; no official benchmark claim", "versions": VERSIONS,
    "source_sha256": source_sha256, "seed": SEED, "selected_rows": len(sample),
    "partition_counts": {p: len(indices[p]) for p in partitions}, "architecture": ARCHITECTURE,
    "feature_schema": FEATURE_SCHEMA, "encoding_audit": encoding_audit,
    "training": {"optimizer": "Adam", "learning_rate": 0.01, "max_epochs": MAX_EPOCHS,
        "patience": PATIENCE, "min_delta": MIN_DELTA, "best_epoch": best_epoch,
        "epochs_run": epoch, "seconds": training_seconds},
    "baseline_protocol": {"ridge_alpha": 1, "Morgan": {"radius": 2, "bits": 1024, "chirality": False},
        "forest_trees": 64, "forest_max_depth": 12}, "test_metrics": metrics.to_dict(orient="index")}
(OUT / "experiment.json").write_text(json.dumps(record, indent=2)+"\n", encoding="utf-8")
print("Saved checkpoint, split, history, predictions and protocol to", OUT)
print("Checkpoint prediction round-trip passed.")

## 12.3.10 Changing the task: classification and multiple assays

The message-passing encoder can be reused while the head, labels, and loss change. Architecture reuse does not imply that labels or evaluation protocols are interchangeable.

| Task | Head output per graph | Loss | Essential evaluation detail |
| --- | --- | --- | --- |
| One continuous property | One unrestricted scalar | MSE or a justified robust loss | Units, MAE/RMSE, outliers, group split |
| One binary assay | One logit | `BCEWithLogitsLoss` (no sigmoid before loss) | Prevalence, PR curve/AP, sensitivity/specificity, calibration |
| Several independent binary assays | One logit per assay | Masked binary loss | Missing labels are not negatives; report each assay |
| Mutually exclusive classes | One logit per class | `CrossEntropyLoss` | Class support and a suitable confusion matrix |

For sparse labels, replace missing values **before** evaluating the elementwise loss; `NaN * 0` is still `NaN`. The following numerical example is a loss-function check, not a trained assay model. Its reduction gives equal weight to each observed graph–assay pair; equal weighting of assays requires a different, explicit reduction.

In [ ]:
illustrative_logits = torch.tensor([[0.4, -0.8], [-0.2, 0.7], [1.2, 0.1]], requires_grad=True)
assay_labels = torch.tensor([[1., float("nan")], [0., 1.], [float("nan"), 0.]])
observed = torch.isfinite(assay_labels)
assert observed.any()
safe_labels = torch.where(observed, assay_labels, torch.zeros_like(assay_labels))
elementwise_loss = nn.functional.binary_cross_entropy_with_logits(illustrative_logits, safe_labels, reduction="none")
masked_loss = elementwise_loss[observed].mean()
masked_loss.backward()
assert torch.isfinite(masked_loss) and torch.all(illustrative_logits.grad[~observed] == 0)
print(f"Loss over {int(observed.sum())} observed labels: {float(masked_loss.detach()):.4f}")

## Exercises and suggested answers

1. The GNN is slower or less accurate than a baseline on this split. What conclusions are justified?
2. Why are both `graph batch` and `sample partition` necessary? Does one replace the other?
3. Why fit the target scale on training rows only? How do we return an MSE-trained standardized model to log solubility units?
4. What experiment would test whether bonds improve prediction over atom composition? Specify how you would prevent test-set tuning.
5. How would you evaluate a GNN intended for molecules synthesized next year? What missing metadata would you request?
6. Why are neither a Tanimoto value nor a highlighted atom a calibrated measure of chemical certainty?

<details><summary>Suggested answers</summary>

1. Report the result for the specified data, split, features, model settings and budget. It does not establish a universal architecture ranking. Investigate learning curves and data quality, then predefine a new experiment with a fresh held-out assessment.
2. Batching offsets node indices so separate molecules can be processed efficiently. Partitioning separates observations for learning, selection, and assessment. A correct batch can still contain leaked train/test data.
3. The target distribution is learned preprocessing. Multiply the standardized prediction by the training standard deviation and add the training mean; RMSE scales by that same standard deviation.
4. Predeclare a no-edge, atom-encoder-plus-readout baseline with the same rows, input features, optimizer budget, and validation protocol. It still knows composition and size. Compare on untouched test groups after selection; a single inference-time edge deletion is an out-of-distribution perturbation, not a trained architecture ablation.
5. Use a justified historical cutoff and keep related series/measurement repeats under control. Request dates, project/series identities, assay conditions, units, replicate identifiers, and any preprocessing history.
6. Similarity describes an encoding-dependent structural relationship. Attribution describes a model's response to its inputs. Neither alone estimates predictive error or establishes a causal molecular mechanism.

</details>

## References and next step

* [Gilmer et al., Neural Message Passing for Quantum Chemistry](https://proceedings.mlr.press/v70/gilmer17a.html): MPNN framework.
* [Yang et al., Analyzing Learned Molecular Representations for Property Prediction](https://arxiv.org/abs/1904.01561): directed-bond models and realistic molecular evaluation.
* [Wu et al., MoleculeNet](https://doi.org/10.1039/C7SC02664A) and [dataset provenance](datasets/README.md): dataset context and the local measured target.
* [RDKit fingerprint generator API](https://www.rdkit.org/docs/source/rdkit.Chem.rdFingerprintGenerator.html) and [PyTorch `index_add_`](https://docs.pytorch.org/docs/2.11/generated/torch.Tensor.index_add_.html): implementation interfaces.
* [scikit-learn common pitfalls](https://scikit-learn.org/stable/common_pitfalls.html): fitted preprocessing and leakage.

[Continue to 12.4: diagnosing and explaining graph models](Chapter12_Part4.ipynb).